## Data Loading 

In [2]:
import pandas as pd

df = pd.read_csv("../data/clean_credit_applications.csv")

df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN


## Executive Summary

The previous analysis identified significant bias in NovaCred’s credit approval system, including gender, age, and geographic location disparities. Rejections for credit approval are primarily justified using a generic “algorithm_risk_score,” which does not explain how decisions are made. These findings raise concerns about fairness, transparency, and accountability.

The purpose of this section is to evaluate these issues from a Governance perspective by mapping them to GDPR and EU AI Act requirements.

 # Privacy and Governance Gaps 

## 1. Sensitive PII stored without protection

To understand how sensitive personal data is handled, the dataset was examined to identify what personal information is stored and whether it is protected.

In [3]:
df[[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address"
]].head()

,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105


The dataset stores Personal Identifiable Information, such as:
 full names, email addresses, Social Security Number (SSNs) and IP addresses in plain text format.

These fields can lead to identification of individuals since they are not hashed or encrypted. This unprotected personal data increases the risk of identity theft or fraud. From a Governance perspective, there is no visible evidence of technical protection measures embedded at the data level, the dataset itself does not demonstrate privacy-by-design safeguards. This suggests weak implementation of preventive data protection controls.


#### Mapping to GDPR - This raises concerns under:

- Article 5(1)(c) – Data Minimization: Personal data must be adequate, relevant, and limited to what is necessary for the stated purpose.

In a credit scoring context, it must be questioned whether storing full SSNs and IP addresses is needed for risk modelling or decision-making. If these identifiers are not essential to the algorithm’s performance, retaining them may constitute excessive data collection and violate the proportionality requirement embedded in data minimization.

- Article 32 – Security of Processing: Organizations must implement appropriate technical and organizational measures to ensure the confidentiality and integrity of personal data. 

Keeping sensitive data in plain text makes it easier to access if the system is breached. Important financial information should be protected using encryption, hashing, and limited access controls.


- Article 25 – Data Protection by Design and by Default: Controllers must integrate privacy safeguards into the system architecture from the outset. 

Storing SSNs and IP addresses in plain text shows that privacy safeguards were not integrated into the system design. A better design would limit access to sensitive data and only make necessary information available.


 #### Recommendation:

NovaCred should implement a layered data protection strategy for sensitive identifiers.These fields (such as SSNs among others) should be pseudonymized (hashed) before being used for any analytical purpose and encrypted at rest to reduce the risk of unauthorized access.

Also, the organization should assess whether storing full SSNs and IP addresses is strictly necessary for credit risk modelling. Where such identifiers are not essential to the model’s performance, they should be removed from operational datasets to ensure compliance with data minimization principles.

Privacy safeguards should be embedded into the system architecture by default, ensuring that sensitive data exposure is reduced at every stage of the data lifecycle.


## Demonstration of Pseudonymization (SSN Hashing)

In [4]:
import hashlib

def hash_value(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

# Create new pseudonymized SSN column
df["pseudonymized_ssn"] = df["applicant_info.ssn"].apply(hash_value)

df[["applicant_info.ssn", "pseudonymized_ssn"]].head()

,applicant_info.ssn,pseudonymized_ssn
0,596-64-4340,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1,425-69-4784,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2,370-78-5178,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3,194-35-1833,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4,480-41-2475,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...



The SSN field has been pseudonymized using a SHA-256 hashing function. This process converts the original identifier into a secure, irreversible string while maintaining consistency across records. In other words, the same SSN will always produce the same hashed value, but the original identifier cannot be directly reconstructed from the hash.

This approach reduces the risk of direct identification of unauthorized access and demonstrates how sensitive identifiers can be protected while preserving their analytical utility. From a governance perspective, pseudonymization supports GDPR requirements related to Security of Processing and Data Protection by Design, as it reduces the exposure of highly sensitive personal data within the system.

However, pseudonymized data remains classified as personal data under the GDPR if it can potentially be re-linked to an individual using additional information. Therefore, pseudonymization alone is not sufficient to eliminate regulatory obligations, the company must also control who can access the data and ensure that stored information is properly protected.



## 2. No Consent Tracking Mechanism


To assess whether personal data processing is legally justified, the dataset was examined for any fields documenting user consent or lawful basis for processing.

In [5]:
df.columns

Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.annual_income',
       'financials.credit_history_months', 'financials.debt_to_income',
       'financials.savings_balance', 'decision.loan_approved',
       'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate',
       'decision.approved_amount', 'notes', 'pseudonymized_ssn'],
      dtype='object')


It can be observed that the dataset has no field documenting user consent or other lawful basis for processing personal data. As such, the processing of sensitive identifiers and financial data raises compliance concerns.


#### Mapping to GDPR - This raises concerns under:

- Article 6 - Lawfulness of Processing: This article states that personal data may only be processed where a valid legal basis exists (for instance: consent, contract, legal obligation).

The absence of any consent or lawful-basis record suggests weak implementation of this requirement.

- Article 5(2) – Accountability: It is required for organizations to demonstrate compliance with GDPR obligations.

Without documented evidence or lawful processing, NovaCred may be unable to justify its data practices to regulators.



#### Recomendation: 

NovaCred should implement a formal lawful-basis documentation framework for all personal data processing activities. The legal basis for each category of data (e.g., contract, legal obligation, consent) should be clearly recorded and maintained.

If consent is relied upon, the system must record consent status, timestamps, and allow withdrawal. These records should be auditable and securely stored to demonstrate compliance with GDPR requirements.

In addition, NovaCred should maintain a documented record of processing activities to ensure that the legal justification for data processing can be verified by supervisory authorities if required.



## 3. Missing Data Retention Policy 

To assess whether personal data is retained in compliance with GDPR requirements, the dataset was examined for any indication of retention limits, deletion timestamps, or lifecycle management controls.

Although the dataset includes a processing timestamp, it does not contain any fields indicating retention limits, deletion dates, or data lifecycle controls. There is no information specifying how long personal data is stored or when it is removed from the system.

#### Mapping to GDPR – This raises concerns under:

- Article 5(1)(e) – Storage Limitation: Personal data must not be kept longer than necessary for the purpose for which it is processed.

Without a clearly defined retention period, NovaCred may be keeping sensitive personal and financial data stored indefinitly. This creates regulatory risk and conflicts with the GDPR requirement that personal data must only be stored for as long as necessary for its intended purpose.

#### Recomendation:

NovaCred should define and document clear retention periods for each category of personal data processed within the credit scoring system. These retention periods should be based on legal, contractual, or regulatory requirements.

Automated deletion, anonymization, or archiving mechanisms should be implemented to ensure that personal data is not stored longer than necessary. In addition, retention policies should be formally documented and regularly reviewed to ensure ongoing compliance with GDPR storage limitation requirements.

## 4. No Audit Trail for Decisions

In order to evaluate the transparency and traceability of automated credit decisions, the dataset was examined to determine whether it includes any records explaining how the model comes to a final decision of approving or rejecting for credit. Since loan approval decisions have serious consequences for individuals, it is a serious matter and it is important to assess whether the system provides sufficient documentation to justify its model decisions.

In [6]:
df[["decision.loan_approved", "decision.rejection_reason"]].head()

,decision.loan_approved,decision.rejection_reason
0,False,algorithm_risk_score
1,False,algorithm_risk_score
2,True,NaN
3,True,NaN
4,False,algorithm_risk_score


As shown in the output, the dataset provides information whether a loan was approved or rejected and gives a general reason (“algorithm_risk_score”) when an application is denied. However, it does not provide any detailed explanation of how the risk score was calculated or which variables influenced the final decision. Since there is no sort of explanation of how the final decision is reached, this suggests the system lacks a structured audit trail capable of explaining automated credit decisions.

Given that prior analysis identified potential bias in approval outcomes, the absence of traceability further increases governance risk, as affected individuals cannot meaningfully understand or challenge the basis of the decision.



#### Mapping to GDPR – This raises concerns under:

- Article 22 – Automated Decision-Making:
This article establishes safeguards when decisions are based only on automated processing and significantly affect individuals. In such cases, individuals must be provided with meaningful information about the logic involved in order to have the ability to contest the decision.

Since the dataset provides only a generic rejection reason (“algorithm_risk_score”) without explaining how the score was calculated or which variables influenced the outcome, the system may not provide sufficient transparency or safeguards required under this article.

- Article 5(1)(a) – Transparency:
Personal data must be processed in a lawful, fair, and transparent manner.

The absence of detailed explanation or traceable reasoning limits the ability of individuals to understand how their data influences automated credit decisions, which may weaken compliance with the transparency principle.


#### Mapping to EU AI Act - This raises concerns under: 

- High-Risk Classification:

Under the EU AI Act, credit scoring systems are considered high-risk because they can significantly affect individuals’ financial opportunities. High-risk systems are subject to stricter governance and oversight requirements.

- Logging and Documentation Requirements:

High-risk AI systems must keep proper records and logs so that decisions can be reviewed and audited. However, the dataset only shows the final outcome and a generic rejection reason (“algorithm_risk_score”), without explaining how the decision was reached. This limits traceability and auditability.

- Human Oversight Requirements:

The EU AI Act requires that high-risk systems allow effective human oversight. Without detailed decision records, it becomes difficult for a human reviewer to understand, assess, or challenge automated credit decisions.

The absence of an audit trail therefore weakens NovaCred’s ability to demonstrate compliance with the EU AI Act’s transparency and oversight obligations


#### Recomendation:


NovaCred should implement prove how it records and explanation mechanisms for all automated credit decisions. The system should store more detailed information about how the calculated risk score is reached, key factors influencing the outcome, and the model version used.
The system should also maintain auditable logs that allow decisions to be reviewed by internal auditors or regulators. In addition, human reviewers should have enough information to understand how the decision was made and be able to change it if necessary.

These measures would strengthen transparency, enable meaningful oversight, and support compliance with both GDPR and EU AI Act requirements.


## 5. Lack of human oversight documentation